# Выбор модели машинного обучения: расчёт и оценка

Этот ноутбук — **и отчёт, и расчёт**: датасет характеристик моделей,
подробные пояснения и код математических операций.

**Задача.** Даны несколько готовых моделей МО. Нужно выбрать **одну
лучшую** и **доказать** выбор.

**Принципы.** Объективно; без весов; без критерия Парето; без датасетов и
экспериментов — оценка извлекается из **архитектуры** модели.

> **Зависимости.** Нужны `numpy`, `pandas`; для графиков — `matplotlib`.
> Установить в то же окружение, что и ядро Jupyter:
> `uv pip install --python /путь/к/окружению/bin/python numpy pandas matplotlib`
> Если `matplotlib` нет — ноутбук всё равно выполнится, графики будут
> пропущены.

## 1. Постановка задачи

Мы выбираем модель для **мобильного робота**, который распознаёт объекты
и показывает результат **оператору**. Нужна модель, которая **успевает**
работать в реальном времени и **влезает** на борт.

Формально это задача принятия решений:

- **Альтернативы** $A_i$ — модели (готовые, мы их не обучаем).
- **Состояния природы** $\theta_j$ — то, что **заранее неизвестно**:
  насколько сложной будет сцена. Берём три сцены: **5, 20, 50** объектов.
- **Выигрыш** $a_{ij}$ — числовая оценка модели $A_i$ в сцене $\theta_j$
  (больше — лучше). Здесь это **достижимая частота кадров**.

**Почему «в условиях неопределённости».** Мы не знаем заранее, в какой
сцене будет работать робот, поэтому сравниваем модели **сразу во всех**
сценах и применяем критерии, не требующие вероятностей.

In [ ]:
import numpy as np
import pandas as pd

try:
    import matplotlib
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception as e:
    HAS_MPL = False
    print('matplotlib недоступен:', e)
    print('Установите: uv pip install --python <env>/bin/python matplotlib')

# --- параметры (можно менять) ---
Pi    = 100.0    # производительность вычислителя, Гоп./с
Ppow  = 10.0     # мощность, Вт
alpha = 1e-3     # стоимость постобработки, Г на объект^2
Tmax  = 0.1      # порог задержки, с (реальное время)
Mmax  = 64.0     # порог памяти, МБ (борт)
states = [5, 20, 50]   # объектов в сцене

# --- датасет моделей ---
rows = [
    # model,       family,           P(млн), F(Г),  mAP,  layers, out
    ("YOLO11n",    "1-stage CNN",    2.62,   6.6,   39.5, 181,    "box"),
    ("YOLOv8n",    "1-stage CNN",    3.16,   8.9,   37.3, 129,    "box"),
    ("YOLO11s",    "1-stage CNN",    9.46,  21.7,   47.0, 181,    "box"),
    ("YOLOv8s",    "1-stage CNN",   11.17,  28.8,   44.9, 129,    "box"),
    ("RT-DETR-l",  "transformer",   32.0,  103.5,   53.0, np.nan, "box"),
    ("FasterRCNN", "2-stage CNN",   41.5,  180.0,   37.4, 100,    "box"),
    ("CNN+LSTM",   "CNN+recurrent",  3.15,   6.61,  39.5, 181,    "box"),
]
df = pd.DataFrame(rows, columns=["model","family","P","F","mAP","layers","out"])
df = df.set_index("model")
df

## 2. Характеристики

Модель описывается **характеристиками**, каждая — это **число с
формулой и единицами**. Группы:

| Группа | Что описывает | Примеры |
| --- | --- | --- |
| A ресурсы | «сколько стоит» модель | $P$, $F$, память $M$, задержка $T$, частота $f$, энергия $Q$ |
| B архитектура | как устроена | глубина $L$, страйд, рецептивное поле |
| C ёмкость | сколько «может» | линейные области, VC-размерность, читаемость |
| D точность (прокси) | что различит | мин. объект, число предсказаний |
| E робототехника | работа на роботе | запас по времени, детерминизм, квантуемость |
| F вывод | что увидит оператор | классы, уверенность, наглядность |
| G веса | что уже есть | датасет предобучения, классы |
| R надёжность | стабильность | детерминизм, устойчивость, калибровка |
| S безопасность | гарантии | гарантированная задержка, отказобезопасность |

Здесь считаем ключевые (ресурсы); остальные участвуют как фильтры и
критерии (см. далее).

### 2.1. Производные характеристики

$$M=4P,\qquad T=\frac{F}{\Pi},\qquad f=\frac{1}{T},\qquad E=\frac{1}{F},\qquad Q=P_{pow}T$$

- $M$ — память под веса (МБ, при $P$ в млн); $T$ — задержка (с);
- $f$ — частота (Гц); $E$ — эффективность; $Q$ — энергия на кадр (Дж).

In [ ]:
df["M_MB"] = 4.0 * df["P"]
df["T"]    = df["F"] / Pi
df["f"]    = 1.0 / df["T"]
df["E"]    = 1.0 / df["F"]
df["Q"]    = Ppow * df["T"]
df[["P","F","M_MB","T","f","E","Q"]].round(3)

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    df["f"].sort_values().plot(kind="barh", ax=ax[0], color="#2c5f8a")
    ax[0].set_title("Частота f, Гц (больше — лучше)")
    df["M_MB"].sort_values().plot(kind="barh", ax=ax[1], color="#c0392b")
    ax[1].set_title("Память весов M, МБ (меньше — лучше)")
    plt.tight_layout(); plt.show()
else:
    print('(графики пропущены: нет matplotlib)')

## 3. Фильтры (отсев)

**Что это.** Жёсткие требования, которые модель обязана выполнить.

**Зачем.** Модель, которая **не успевает** по времени или **не влезает**
в память, для робота бесполезна — её сразу исключаем.

**Условия:** $T\le T_{\max}$ (реальное время), $M\le M_{\max}$ (борт),
тип выхода — рамка.

In [ ]:
df["pass_T"]   = df["T"]    <= Tmax
df["pass_M"]   = df["M_MB"] <= Mmax
df["pass_out"] = df["out"] == "box"
df["admissible"] = df[["pass_T","pass_M","pass_out"]].all(axis=1)
admissible = df.index[df["admissible"]].tolist()
print("Допустимые модели:", admissible)
df[["T","M_MB","out","admissible"]].round(3)

## 4. Выигрыш: достижимая частота

$$f_{ij}=\frac{\Pi}{F_i+\alpha N_j^2}$$

**Что означает.** Сколько кадров в секунду успеет модель $i$ в сцене
$\theta_j$ с числом объектов $N_j$.

**Почему растёт стоимость.** После сети результат дорабатывается
(например, удаляются дубли рамок — NMS). Рамки сравнивают **попарно**,
поэтому стоимость растёт как $N^2$: для 5 объектов — 0.025 Г, для 50 —
2.5 Г (сопоставимо с самими моделями).

**Практически.** В густой сцене падает частота — это и есть
«неопределённость», которую мы учитываем.

In [ ]:
# пример одного значения: YOLO11n, theta=5
print("Пример: 100 / (6.6 + 0.001*5^2) =", round(100/(6.6+0.001*25), 3))

def payoff_matrix(df_sub, states, alpha=alpha, Pi=Pi):
    return pd.DataFrame(
        {f"theta={N}": Pi/(df_sub["F"] + alpha*N**2) for N in states},
        index=df_sub.index,
    )

A = payoff_matrix(df.loc[admissible], states)
A.round(3)

In [ ]:
if HAS_MPL:
    A.plot(kind="bar", figsize=(9,4), color=["#7fb3d5","#2c5f8a","#14395c"])
    plt.title("Частота по сценам (5 / 20 / 50 объектов)")
    plt.ylabel("Гц"); plt.xticks(rotation=0)
    plt.legend(title="сцена"); plt.tight_layout(); plt.show()

## 5. Критерии принятия решений в условиях неопределённости

Каждый критерий по-своему сводит матрицу к одному числу на модель.

| Критерий | Выбор | Идея |
| --- | --- | --- |
| Лаплас | $\max_i \bar a_i$ | сцены равновероятны |
| Оптимизм | $\max_i \max_j a_{ij}$ | лучший случай |
| Вальд | $\max_i \min_j a_{ij}$ | худший случай |
| Сэвидж | $\min_i \max_j r_{ij}$ | минимум сожаления |
| Гурвиц | $\max_i[\alpha\max_j+(1-\alpha)\min_j]$ | компромисс |

### 5.1. Критерий Лапласа

**Идея.** Считаем все сцены **одинаково возможными** и берём модель с
лучшим **средним**. «Нейтральный» взгляд.

**Почему.** Мы не знаем сцен заранее и не даём ни одной преимущества.

**Практика.** Обычная эксплуатация, когда сцены разные и заранее
неизвестны.

In [ ]:
def laplace(A): return A.mean(axis=1)
lap = laplace(A)
print(lap.round(3).to_string())
print("Победитель (Лаплас):", lap.idxmax())
if HAS_MPL:
    lap.sort_values().plot(kind="barh", figsize=(7,2.5), color="#2c5f8a")
    plt.title("Лаплас: среднее по сценам"); plt.show()

### 5.2. Критерий оптимизма (максимакс)

**Идея.** Расчёт на **лучший** случай: у каждой модели её максимум, берём
наибольший. «Азартный» взгляд.

**Почему.** Иногда важна **пиковая** скорость, а редкие просадки
допустимы.

**Практика.** Разовые осмотры, простые сцены, запас по времени.

In [ ]:
def optimism(A): return A.max(axis=1)
opt = optimism(A)
print(opt.round(3).to_string())
print("Победитель (оптимизм):", opt.idxmax())

### 5.3. Критерий пессимизма Вальда (максимин)

**Идея.** Готовимся к **худшему** случаю: у каждой модели её минимум,
берём лучший из худших. «Осторожный» взгляд.

**Почему.** Гарантирует минимум, важный для управления.

**Практика.** Контур реального времени и безопасность, где просадка
опасна.

In [ ]:
def wald(A): return A.min(axis=1)
wal = wald(A)
print(wal.round(3).to_string())
print("Победитель (Вальд):", wal.idxmax())

### 5.4. Критерий Сэвиджа (минимакс сожаления)

**Идея.** «Сожаление» — потеря против лучшей модели в той же сцене.
Берём модель с минимальным **худшим** сожалением.

**Почему.** Защита от «упущенной выгоды»: нигде не оказаться сильно хуже.

**Практика.** Неизвестные условия — не проиграть другим где угодно.

In [ ]:
def regret(A): return (A.max(axis=0) - A).max(axis=1)
reg = regret(A)
print(reg.round(3).to_string())
print("Победитель (Сэвидж):", reg.idxmin())
if HAS_MPL:
    plt.figure(figsize=(6,2.5))
    plt.imshow((A.max(axis=0)-A).round(2), cmap="Reds", aspect="auto")
    plt.colorbar(label="сожаление"); plt.yticks(range(len(A)), A.index)
    plt.xticks(range(len(A.columns)), A.columns)
    plt.title("Матрица сожалений"); plt.tight_layout(); plt.show()

### 5.5. Критерий Гурвица

**Идея.** Компромисс между оптимизмом и пессимизмом с коэффициентом
$\alpha\in[0,1]$: $\alpha<0.5$ — осторожно, $>0.5$ — с надеждой.

**Почему.** Позволяет **настроить** характер решения под задачу.

**Практика.** Для безопасности брать $\alpha<0.5$, для
производительности — $>0.5$.

In [ ]:
def hurwitz(A, a): return a*A.max(axis=1) + (1-a)*A.min(axis=1)
for a in (0.3, 0.5, 0.8):
    h = hurwitz(A, a)
    print(f"alpha={a}: победитель {h.idxmax()}, значения {h.round(2).to_dict()}")
if HAS_MPL:
    alphas = np.linspace(0, 1, 101)
    plt.figure(figsize=(7,3))
    for m in A.index:
        plt.plot(alphas, [hurwitz(A, a)[m] for a in alphas], label=m)
    plt.xlabel("alpha (оптимизм)"); plt.ylabel("H"); plt.legend()
    plt.title("Гурвиц: H(alpha) по моделям"); plt.tight_layout(); plt.show()

## 6. Подсчёт баллов (итоговый выбор)

Чтобы свести все критерии в одно решение, начисляем **баллы за место**:
1-е место — $n$, …, последнее — 1. Сумма:
$$\pi_i^{(k)}=n-r_i^{(k)}+1,\qquad P_i=\sum_k \pi_i^{(k)},\qquad
A^\ast=\arg\max_i P_i.$$

**Зачем.** Один числовой итог вместо нескольких разных ответов.

In [ ]:
def points(values, lower_is_better=False):
    r = values.rank(ascending=lower_is_better, method="min")
    return len(values) - r + 1

total = (points(laplace(A)) + points(optimism(A)) + points(wald(A))
         + points(regret(A), lower_is_better=True) + points(hurwitz(A, 0.5)))
total = total.sort_values(ascending=False)
print(total.to_string())
print("Победитель:", total.idxmax())
if HAS_MPL:
    total.plot(kind="bar", figsize=(6,3), color="#14395c")
    plt.title("Баллы моделей (сумма по критериям)"); plt.ylabel("баллы")
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 7. Вариант II — баланс «скорость/точность»

**Идея.** Учесть и **точность**, и **скорость**: очень точная, но
медленная модель может быть хуже. Выигрыш
$a_{ij}=\dfrac{\text{mAP}_i\cdot f_{ij}}{100}$.

**Практика.** Показывает «полезность» — сколько точности в секунду
получает оператор.

In [ ]:
B = A.mul(df.loc[admissible, "mAP"], axis=0) / 100.0
B.round(3)

In [ ]:
res2 = {
    "Лаплас":   laplace(B).idxmax(),
    "Оптимизм": optimism(B).idxmax(),
    "Вальд":    wald(B).idxmax(),
    "Сэвидж":   regret(B).idxmin(),
    "Гурвиц":   hurwitz(B, 0.5).idxmax(),
}
print(pd.Series(res2).to_string())

## 8. Вариант III — показатель $\Phi$

**Идея.** Свести характеристики в один показатель без весов: нормировать
каждую к $[0,1]$ (по данным) и усреднить.
$$z_c(m)=\frac{\delta_c x_c(m)-\min \delta_c x_c}{\max \delta_c x_c-\min \delta_c x_c},\qquad
\Phi(m)=\frac1{|C|}\sum_c z_c(m)$$

In [ ]:
chars = [("P", False), ("F", False), ("M_MB", False), ("T", False),
         ("f", True), ("E", True), ("Q", False)]

def phi(df_sub, chars):
    z = pd.DataFrame(index=df_sub.index)
    for name, bigger in chars:
        x = df_sub[name]; lo, hi = x.min(), x.max()
        z[name] = (x-lo)/(hi-lo) if bigger else (hi-x)/(hi-lo)
    return z.mean(axis=1)

phi_vals = phi(df.loc[admissible], chars).sort_values(ascending=False)
print(phi_vals.round(3).to_string())
print("Победитель по Phi:", phi_vals.idxmax())
if HAS_MPL:
    phi_vals.plot(kind="bar", figsize=(6,3), color="#2c5f8a")
    plt.title("Показатель Phi (больше — лучше)"); plt.ylabel("Phi")
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 9. Вывод

**Что получилось.** Все пять критериев и все три варианта выигрыша
указывают на **одну и ту же** модель — значит выбор **устойчив** и не
зависит от метода.

**Доказательства (числа):**
1. самая высокая частота (в т.ч. в сложной сцене);
2. самая маленькая память и энергия;
3. нулевое сожаление — нигде не проигрывает;
4. максимальные баллы и максимальный $\Phi$.

**Практика.** Выбранная модель даёт $\ge10$ Гц даже в сложной сцене,
экономит память и энергию, к ней добавляется трекинг.

**Проверка — в симуляции** (Gazebo): условия повторяемы, разметка
точна — можно измерить задержку, калибровку (ECE) и устойчивость к шуму
напрямую.

In [ ]:
print("Допустимые модели:      ", admissible)
print("Баллы (по 5 критериям): ", total.round(1).to_dict())
print("Победитель по баллам:   ", total.idxmax())
print("Победитель по Phi:      ", phi_vals.idxmax())
print("Вариант II (баланс):    ", set(res2.values()))